# GPU-Accelerated MAV Visualization in Python  
### From Matplotlib 3D to an Efficient OpenGL Pipeline

This notebook walks through how to replace a CPU-bound Matplotlib 3D animation with a **GPU-backed renderer** while keeping your MAV simulation code in Python.

### What you’ll build
- A static aircraft mesh (uploaded once to the GPU)
- A pose → 4×4 model transform (updated each frame)
- A camera view + perspective projection
- A renderer (ModernGL) and window loop (pyglet)
- A decoupled **simulation step rate** and **render rate** for smooth visuals

> Tip: Keep explanations in Markdown cells and code in Code cells. Run Markdown cells to render them.


## 0) Install dependencies

If you’re in a conda env, run this cell once. You may also need OS packages for OpenGL depending on your Linux setup.

- `pyglet` creates the window and event loop
- `moderngl` provides a clean OpenGL API in Python


In [ ]:
# If needed, install packages (run once)
# !pip install pyglet moderngl numpy

## 1) Why Matplotlib 3D is slow for animation

Matplotlib 3D is convenient but inefficient for real-time 3D:
- It’s mostly CPU-driven (not true GPU rendering)
- Many examples rebuild geometry every frame (allocations + Python overhead)
- It does not use the same optimized render path as real-time engines

What matters:
1. Uploads geometry once
2. Updates a small amount of state each frame (a transform matrix)
3. Lets the GPU handle transformation + rasterization


## 2) Architecture overview

```
┌────────────┐
│  Dynamics  │  ← RK4 + FaM
└─────┬──────┘
      │ state (pn, pe, pd, φ, θ, ψ)
┌─────▼──────┐
│  Renderer  │  ← OpenGL via moderngl
└─────┬──────┘
      │ GPU
┌─────▼──────┐
│  Window    │  ← pyglet event loop
└────────────┘
```

Implement:
- a static aircraft mesh (model/body coordinates)
- pose → model matrix
- view + projection matrices
- a render loop decoupled from sim loop


## 3) Build a static aircraft mesh (model space)

Instead of regenerating faces each frame, we define:
- `V`: vertices in **body coordinates**
- `idx`: triangle indices (triangulated faces)

This mesh stays constant. Only the pose changes.


In [ ]:
import numpy as np

def aircraft_model_mesh(scale=5.0):
    # A compact mesh based plane points.
    # Expand/replace with full vertex list as needed.
    fuse_l1, fuse_l2, fuse_l3 = 2.0, 0.1, 5.0
    fuse_w, fuse_h = 1.0, 1.0

    V = scale * np.array([
        [ fuse_l1, 0.0, 0.0],                    # 0 nose
        [ fuse_l2,  fuse_w/2, -fuse_h/2],         # 1
        [ fuse_l2, -fuse_w/2, -fuse_h/2],         # 2
        [ fuse_l2, -fuse_w/2,  fuse_h/2],         # 3
        [ fuse_l2,  fuse_w/2,  fuse_h/2],         # 4
        [-fuse_l3, 0.0, 0.0],                     # 5 tail
    ], dtype=np.float32)

    # Triangulated faces
    faces = [
        [0, 1, 2], [0, 2, 3], [0, 3, 4], [0, 4, 1],
        [1, 2, 5], [2, 3, 5], [3, 4, 5], [4, 1, 5],
    ]
    idx = np.array(faces, dtype=np.uint32).reshape(-1)
    return V, idx

V, idx = aircraft_model_mesh(scale=5.0)
V.shape, idx.shape

## 4) Pose → model matrix (Euler angles)

Your simulation uses NED:
- position: `(pn, pe, pd)`
- attitude: `(phi, theta, psi)`

We build a 4×4 homogeneous transform:
- top-left 3×3 = rotation matrix `R_b2i`
- top-right 3×1 = translation `[pn, pe, pd]`

> Note: Euler angles have a singularity at `theta = ±90°`. For aggressive aerobatics, quaternions are better.


In [ ]:
def euler_to_R(phi, theta, psi):
    c, s = np.cos, np.sin
    return np.array([
        [c(theta)*c(psi), s(phi)*s(theta)*c(psi)-c(phi)*s(psi), c(phi)*s(theta)*c(psi)+s(phi)*s(psi)],
        [c(theta)*s(psi), s(phi)*s(theta)*s(psi)+c(phi)*c(psi), c(phi)*s(theta)*s(psi)-s(phi)*c(psi)],
        [-s(theta),       s(phi)*c(theta),                      c(phi)*c(theta)]
    ], dtype=np.float32)

def model_matrix_from_state(pn, pe, pd, phi, theta, psi):
    R = euler_to_R(phi, theta, psi)
    M = np.eye(4, dtype=np.float32)
    M[:3, :3] = R
    M[:3,  3] = np.array([pn, pe, pd], dtype=np.float32)
    return M

# Example
M = model_matrix_from_state(10.0, 0.0, 0.0, 0.1, 0.05, 0.0)
M

## 5) Camera: view matrix + projection matrix

We need:
- **View matrix**: world → camera (where the camera is looking from)
- **Projection matrix**: camera → clip space (perspective)

A simple approach is a chase camera that looks at the aircraft position.


In [ ]:
def perspective(fovy_deg, aspect, znear, zfar):
    f = 1.0 / np.tan(np.deg2rad(fovy_deg) / 2.0)
    M = np.zeros((4, 4), dtype=np.float32)
    M[0, 0] = f / aspect
    M[1, 1] = f
    M[2, 2] = (zfar + znear) / (znear - zfar)
    M[2, 3] = (2 * zfar * znear) / (znear - zfar)
    M[3, 2] = -1.0
    return M

def look_at(eye, target, up):
    eye = np.array(eye, dtype=np.float32)
    target = np.array(target, dtype=np.float32)
    up = np.array(up, dtype=np.float32)

    f = target - eye
    f /= (np.linalg.norm(f) + 1e-9)
    r = np.cross(f, up)
    r /= (np.linalg.norm(r) + 1e-9)
    u = np.cross(r, f)

    M = np.eye(4, dtype=np.float32)
    M[0, :3] = r
    M[1, :3] = u
    M[2, :3] = -f
    M[:3, 3] = -M[:3, :3] @ eye
    return M

# Example view/proj
eye = np.array([-80, -80, -30], dtype=np.float32)
target = np.array([0, 0, 0], dtype=np.float32)
up = np.array([0, 0, -1], dtype=np.float32)  # opposite Down in NED

Vw = look_at(eye, target, up)
Pw = perspective(60.0, aspect=1000/800, znear=0.1, zfar=5000.0)
Vw, Pw

## 6) Renderer: upload mesh once, draw with an MVP uniform

ModernGL setup:
- Create buffers for vertices/indices
- Define a minimal shader
- Each frame compute: `MVP = P @ V @ M`
- Update uniform and draw

This is the **performance-critical** change: *no per-frame mesh rebuilds*.


In [ ]:
# This cell defines the Renderer class. Run it once.
import moderngl

class Renderer:
    def __init__(self, ctx: moderngl.Context, vertices: np.ndarray, indices: np.ndarray):
        self.ctx = ctx
        self.ctx.enable(moderngl.DEPTH_TEST)

        self.prog = self.ctx.program(
            vertex_shader="""
                #version 330
                uniform mat4 u_mvp;
                in vec3 in_pos;
                void main() {
                    gl_Position = u_mvp * vec4(in_pos, 1.0);
                }
            """,
            fragment_shader="""
                #version 330
                out vec4 f_color;
                void main() {
                    f_color = vec4(0.6, 0.75, 0.95, 1.0);
                }
            """,
        )

        vbo = self.ctx.buffer(vertices.tobytes())
        ibo = self.ctx.buffer(indices.tobytes())
        self.vao = self.ctx.vertex_array(self.prog, [(vbo, "3f", "in_pos")], index_buffer=ibo)

        self.u_mvp = self.prog["u_mvp"]

        # Camera defaults
        self.fovy = 60.0
        self.znear = 0.1
        self.zfar = 5000.0
        self.up = np.array([0, 0, -1], dtype=np.float32)

    def draw(self, width: int, height: int, model_mat: np.ndarray):
        self.ctx.clear(0.05, 0.06, 0.08)
        aspect = width / max(height, 1)

        pos = model_mat[:3, 3]
        eye = pos + np.array([-80, -80, -30], dtype=np.float32)
        target = pos

        Vmat = look_at(eye, target, self.up)
        Pmat = perspective(self.fovy, aspect, self.znear, self.zfar)

        mvp = Pmat @ Vmat @ model_mat
        # OpenGL expects column-major; transpose for contiguous write
        self.u_mvp.write(mvp.T.tobytes())

        self.vao.render()

## 7) Window + timing: decouple sim rate from render rate

We will use:
- `sim_hz` (e.g. 200 Hz)
- `render_hz` (e.g. 60 Hz)

The sim loop uses a fixed timestep accumulator.
The render loop forces a redraw at a fixed rate.

**Important**: pyglet doesn’t use `invalidate()` for redraw scheduling; we schedule rendering explicitly.


In [ ]:
# This cell defines the SimWindow class. Run it once.
import pyglet
import time

class SimWindow(pyglet.window.Window):
    def __init__(self, sim_step_func, get_pose_func, renderer_factory,
                 width=1000, height=800, render_hz=60, sim_hz=200):
        super().__init__(width=width, height=height, caption="MAV Viewer (GPU)", resizable=True)

        self.glctx = moderngl.create_context()
        self.renderer = renderer_factory(self.glctx)

        self.sim_step = sim_step_func
        self.get_pose = get_pose_func

        self.sim_dt = 1.0 / sim_hz
        self.accum = 0.0
        self.last = time.perf_counter()

        pyglet.clock.schedule(self._tick)
        pyglet.clock.schedule_interval(self._render, 1.0 / render_hz)

    def _tick(self, _dt):
        now = time.perf_counter()
        frame_dt = now - self.last
        self.last = now

        self.accum += frame_dt
        if self.accum > 0.25:
            self.accum = 0.25

        while self.accum >= self.sim_dt:
            self.sim_step(self.sim_dt)
            self.accum -= self.sim_dt

    def _render(self, _dt):
        self.dispatch_event("on_draw")
        self.flip()

    def on_draw(self):
        self.clear()
        pn, pe, pd, phi, theta, psi = self.get_pose()
        model = model_matrix_from_state(pn, pe, pd, phi, theta, psi)
        self.renderer.draw(self.width, self.height, model)

## 8) Hook in your simulation (dynamics)

You already have a `dynamics` class with `state` and `update(u)`.

The only things the viewer needs:
- `sim_step(dt)` : advance simulation
- `get_pose()` : return `(pn, pe, pd, phi, theta, psi)`

Below is a template. **Edit the imports** to match your project layout.


In [ ]:
# Template hookup: adjust imports/paths to your project.
# If your project already sets sys.path to include the src folder, you can import directly.

# from dynamics import dynamics
# from params import params

# dyn = dynamics()
# P = params()

# def control_input():
#     # simplest: fixed trim/steady input
#     return P.u_star

# def sim_step(dt):
#     # If your dynamics uses dyn.Ts internally, you can either:
#     #   (a) set dyn.Ts = dt once, OR
#     #   (b) ignore dt and call dyn.update(u) at your preferred sim_hz.
#     u = control_input()
#     dyn.update(u)

# def get_pose():
#     s = dyn.state
#     return (float(s[0,0]), float(s[1,0]), float(s[2,0]),
#             float(s[6,0]), float(s[7,0]), float(s[8,0]))

# def renderer_factory(ctx):
#     return Renderer(ctx, V, idx)

# win = SimWindow(sim_step, get_pose, renderer_factory, render_hz=60, sim_hz=200)
# pyglet.app.run()

## 9) Debug checklist (common issues)

### Window opens but nothing is visible
- Ensure the camera is pointed at the aircraft
- Ensure near/far planes are reasonable
- Confirm `pd` sign convention (NED uses positive down)

### `moderngl.create_context()` fails
- You may be missing OpenGL drivers/libraries on Linux
- Ensure you’re not running over a remote session without GPU acceleration

### Animation is choppy
- Increase `sim_hz` or interpolate pose between sim steps
- Keep rendering fixed at 60 Hz

### Euler singularity
- If you pitch near ±90°, switch to quaternions for attitude propagation & rendering
